# មេរៀនទី 12 - ការកាត់បន្ថយប្រវត្តិការជជែកជាមួយAgent Scratchpad

កំណត់ត្រានេះបង្ហាញពីរបៀបគ្រប់គ្រង context ក្នុងការពិភាក្សាយូរអង្វែងដោយប្រើ Microsoft Agent Framework។ ពេលដែលការជជែកកើនឡើង ចំនួនតូ កិនក៏កើនដែរ — ចុងក្រោយលើសពីបរិវេណ context របស់ម៉ូដែល។ យើងដោះស្រាយវាដោយប្រើ **គំរូសង្ខេប context** និង **agent scratchpad** សម្រាប់ចងចាំយូរ។

## អ្វីដែលអ្នកនឹងរៀន៖
1. **ហេតុអ្វីការគ្រប់គ្រង Context មានសារៈសំខាន់**៖ យល់ដឹងពីកំណត់តួកិន និងបរិវេណ context
2. **Agent ដែលយល់ context**៖ បង្កើត agent ដែលគ្រប់គ្រង context នៃការជជែករបស់ខ្លួន
3. **គំរូសង្ខេប Context**៖ ប្រើឧបករណ៍ដើម្បីបញ្ចុះប្រវត្តិការជជែក
4. **Agent Scratchpad**៖ ចងចាំដែលនៅផ្សេងទៀតបន្ទាប់ពីកាត់បន្ថយ context

## ការត្រៀមខ្លួន៖
- ការតំឡើង Azure OpenAI ជាមួយអថេរបរិដ្ឋានដែលបានកំណត់
- យល់ដឹងពីគំនិត agent មូលដ្ឋានពីមេរៀនមុន


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ហេតុអ្វីបានជា ការគ្រប់គ្រងបរិបទមានសារៈសំខាន់

LLM ដ Каж់មួយមាន **បង្អួចបរិបទ** ដែលមានអតិបរមាចំនួនតូកែនដែលវាអាចដំណើរការបានក្នុងការស្នើសុំមួយ។ ក្រោយពេលកិច្ចสนច្ចាជាច្រើនជំហានដំណើរការ៖

- **ចំនួនតូកែនកើនឡើងតាមបន្ទាត់** ជាមួយសាររបស់អ្នកប្រើប្រាស់និងចម្លើយពីជំនួយការ។
- **តូកែនសួររំពឹងគ្រប់ដំណើរចំណាយ** ពីព្រោះប្រវត្តិប្រវត្តិគ្រប់មួយត្រូវបានផ្ញើឡើងវិញក្នុងគ្រប់ជំហាន។
- ចុងក្រោយ កិច្ចสน  ุญសារនោះ **លើសពីបង្អួចបរិបទ** ហើយម៉ូដែលអាចចុះបញ្ចប់ឬកើតកំហុស។

### យុទ្ធសាស្រ្តសម្រាប់គ្រប់គ្រងបរិបទ

| យុទ្ធសាស្រ្ត | វាដំណើរការ​ដូចម្តេច | ការប្រ_tradeoff |
|---|---|---|
| **ការកាត់បន្ថយ** | ទម្លាក់សារ​ចាស់ៗ | ផ្ដោតបាត់បង់បរិបទដើម |
| **ការសង្ខេប** | សង្ខេបសារចាស់ៗទៅជាសង្ខេប | មានការបាត់បង់ព័ត៌មានខ្លះ ប៉ុន្តែរក្សាចំណុចសំខាន់ |
| **Scratchpad / ម៉ោនជំនួយខាងក្រៅ** | រក្សាទុកព័ត៌មានសំខាន់​ខាងក្រៅកិច្ចสน  ุญ | ត្រូវការហៅឧបករណ៍ ប៉ុន្តែអាចរក្សាទុកបានទាំងអស់ |

ក្នុងកំណត់ត្រានេះ យើងបញ្ចូល **ការសង្ខេប** ជាមួយ **ឧបករណ៍ scratchpad** ដើម្បីឱ្យភ្នាក់ងារអាចរក្សាថ្មីឡើងវិញបណ្តាញខ្សែទិន្នន័យ ទោះបីជាបរិបទកិច្ចสน  ุญត្រូវបានសង្ខេប។


## ការបង្កើតភ្នាក់ងារដែលយល់ដឹងពីបរិបទ


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## ការសម័យភាពការពិភាក្សាយូរ

មកពិលតាមការពិភាក្សាច្រើនជុំ ដើម្បីមើលថាContext ធ្លាប់បន្តគ្នា។ អ្នកតំណាងគួរតែរក្សារពត៌មានសំខាន់ៗ (ចំណូលចិត្ត, បរិមាណថវិកា, កាលបរិច្ឆេទធ្វើដំណើរ) នៅក្នុងជួរពិភាក្សា ហើយបង្ហាញពីការតភ្ជាប់ជា​​​ជាបន្តបន្ទាប់។  


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

សូមយកចិត្តទុកដាក់ទៅលើរបៀបដែលភ្នាក់ងារ​រក្សាទុកបរិបទពីការប្រាស្រ័យទាក់ទងមុនៗ — វាដឹងអំពីជប៉ុន, ស៊ូស៊ី, វត្ត, បទថតរូប, ប្រាក់ប៉ុន្មាន $3000, ការធ្វើដំណើរតែម្នាក់ឯង, និងពេលវេលាឆ្នាំមេសា។ ក្នុងការសន្ទនាខ្លីនេះ វាជាអ្វីដែលដំណើរការល្អ, ប៉ុន្តិពេលសន្ទនាធំជាងនេះ ប្រវត្តិពេញលេញក្លាយទៅជារឿងថ្លៃខ្ពស់ក្នុងការផ្ទេរម្តងទៀត។

មកបន្តការសន្ទនាជាមួយការត្រឡប់ទៀតជាច្រើន ដើម្បីឃើញការចម្រាញ់បរិបទ៖


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## លំនាំសង្ខេបបរិបទ

នៅពេលការសន្ទនាធំឡើង យើងអាចប្រើ **ឧបករណ៍សង្ខេប** ដើម្បីសង្ខេបបរិបទដែលបានប្រមូលផ្តុំទៅជា រចនាសម្ព័ន្ធតូច ។ មន្ត្រីហៅឧបករណ៍នេះដើម្បីរក្សាទុកចំណូលចិត្តសំខាន់ៗ ដូច្នេះ បើសារ​ចាស់ៗ​ត្រូវបានលុបចោល ការទិន្នន័យសំខាន់ៗនឹងត្រូវបានរក្សាទុក។

លំនាំនេះគឺជាគ្រឹះដើម្បីកាត់បន្ថយប្រវត្តិដែលមានភាពចំរូងចំរាស៖
1. មន្ត្រីកំណត់ការពិតសំខាន់ៗពីការសន្ទនា
2. វាហៅឧបករណ៍សង្ខេបដើម្បីរក្សាទុកវា
3. សារចាស់ៗអាចត្រូវបានលុបដោយសុវត្ថិភាព ពីព្រោះសេចក្ដីសង្ខេបបានចាប់យកអ្វីដែលមានសារៈសំខាន់

ខាងក្រោមនេះ យើងកំណត់ឧបករណ៍ `summarize_preferences` ដែលមន្ត្រីអាចហៅដើម្បីកត់ត្រាសេចក្ដីសង្ខេបតូចមួយនៃអ្វីដែលវាបានរៀន។


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីវិធីគ្រប់គ្រងបរិបទក្នុងការសន្ទនាជាមួយភ្នាក់ងារដែលរយៈពេលវែងដោយប្រើ Microsoft Agent Framework៖

### គ念សំខាន់ៗ
- **ជញ្ជាំងបរិបទមានដែនកំណត់** — រាល់តួអក្សរនៅក្នុងប្រវត្តិសន្ទនាគឺមានថ្លៃដើម និងគិតទៅកាន់ដែនកំណត់។
- **ឧបករណ៍សង្ខេប** ឲ្យភ្នាក់ងារប្រមូលបរិបទបានក្នុងល្វែងខ្លីៗ ដែលកាត់បន្ថយការប្រើតួអក្សរប៉ុន្តែមិនបាត់បង់ព័ត៌មានសំខាន់ទេ។
- **សន្លឹកសរសេរភ្នាក់ងារ** ផ្ដល់សមនុស្សនិងអង្គចងចាំខាងក្រៅដែលនៅជាប់ខ្លួនទាំងពេលខ្លះមានការកាត់បន្ថយក្នុងសន្ទនា។

### អ្វីដែលអ្នកបានបង្កើត
- **ភ្នាក់ងារយល់ដឹងបរិបទ** ដែលរក្សា​​ភាពអចលន៍ក្នុងសន្ទនាច្រើនជំហាន
- **ឧបករណ៍សង្ខេប** (`summarize_preferences`) ដែលកត់ត្រាព័ត៌មានសំខាន់របស់អ្នកប្រើជាលំនាំខ្លីៗ
- **សន្ទនាជាច្រើនជំហាន** បង្ហាញការរក្សាបរិបទ និងការដោះស្រាយការផ្លាស់ប្តូរ

### ការអនុវត្តជាការពិត
- **រ៉ូបូតបម្រើអតិថិជន**៖ ចងចាំចំណង់ចំណូលចិត្តនៅក្នុងកំឡុងពេលគាំទ្រខ្លីវែង
- **ជំនួយការផ្ទាល់ខ្លួន**៖ តាមដានគម្រោងដំណើរការដោយមិនចាំបាច់អះអាងបរិបទឡើងវិញ
- **គ្រូបង្រៀន**៖ រក្សាកម្រិតនៃកំណត់ត្រានិស្សិតច្រើនជាមួយផ្លូវការសន្ទនា

### ជំហានបន្ទាប់
- អនុវត្តឧបករណ៍សន្លឹកសរសេរពេញលេញជាមួយអង្គចងចាំផ្ទុកជាឯកសារ
- បន្ថែមការកាត់បន្ថយប្រវត្តិដោយស្វ័យប្រវត្តិបន្ទាប់ពីសង្ខេប
- បញ្ចូលជាមួយមូលដ្ឋានទិន្នន័យវ៉ិចទ័រសម្រាប់ស្វែងរកអង្គចងចាំន័យ
- សាងសង់ភ្នាក់ងារដែលអាចបន្តសន្ទនារយៈពេលប៉ុន្មានថ្ងៃក្រោយជាមួយបរិបទពេញលេញ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
